In [6]:
import pandas as pd
from pyecharts.charts import Bar, Timeline, Map, Line, Pie, Scatter, Page
from pyecharts import options as opts
from pyecharts.globals import ThemeType
from pyecharts.commons.utils import JsCode

# ===================== 全局统一样式配置 =====================
GLOBAL_WIDTH = "1000px"
GLOBAL_HEIGHT = "550px"
GLOBAL_THEME = ThemeType.MACARONS
TITLE_FONT_SIZE = 18
TITLE_COLOR = "#2c3e50"
AXIS_FONT_SIZE = 11
LABEL_FONT_SIZE = 10
TOOLTIP_STYLE = opts.TooltipOpts(trigger="item", textstyle_opts=opts.TextStyleOpts(font_size=11))

# 读取公共数据源
df = pd.read_excel("../data/近10年各省份居民收支与人口数据.xlsx")
df.columns = df.columns.str.strip()

# ===================== 1. 动态条形图：收入TOP10排名 =====================
timeline_bar = (
    Timeline(init_opts=opts.InitOpts(width=GLOBAL_WIDTH, height=GLOBAL_HEIGHT, theme=GLOBAL_THEME))
    .add_schema(
        orient="vertical",
        play_interval=1000,
        pos_right="null",
        pos_top="20",
        pos_bottom="20",
        width="60px",
        is_auto_play=True,
        is_loop_play=False,
        label_opts=opts.LabelOpts(position="left", font_size=LABEL_FONT_SIZE)
    )
)

for year in range(2016, 2026):
    year_data = df[df["年份"] == year].sort_values(by="人均可支配收入（元）", ascending=False).head(10).reset_index(drop=True)
    year_data = year_data.sort_values(by="人均可支配收入（元）", ascending=True).reset_index(drop=True)

    bar = (
        Bar(init_opts=opts.InitOpts(width=GLOBAL_WIDTH, height=GLOBAL_HEIGHT, theme=GLOBAL_THEME))
        .add_xaxis(year_data["省份名称"].tolist())
        .add_yaxis("人均可支配收入（元）", [int(x) for x in year_data["人均可支配收入（元）"].tolist()])
        .reversal_axis()
        .set_global_opts(
            title_opts=opts.TitleOpts(
                title=f"全国{year}年人均可支配收入TOP10省份排名",
                title_textstyle_opts=opts.TextStyleOpts(font_size=TITLE_FONT_SIZE, color=TITLE_COLOR)
            ),
            yaxis_opts=opts.AxisOpts(
                name="省份名称",
                axislabel_opts=opts.LabelOpts(font_size=AXIS_FONT_SIZE)
            ),
            xaxis_opts=opts.AxisOpts(
                name="人均可支配收入（元）",
                min_=20000, max_=100000,
                axislabel_opts=opts.LabelOpts(font_size=AXIS_FONT_SIZE)
            ),
            tooltip_opts=TOOLTIP_STYLE
        )
        .set_series_opts(
            label_opts=opts.LabelOpts(is_show=True, position="right", font_size=LABEL_FONT_SIZE)
        )
    )
    timeline_bar.add(bar, time_point=str(year))

# ===================== 2. 动态地图：收入地理分布（统一青绿色系） =====================
df["收支差额（元）"] = df["人均可支配收入（元）"] - df["人均消费支出（元）"]
df_map = df[(df["年份"].between(2016, 2025)) & (df["人均可支配收入（元）"].notna())].copy()
df_map = df_map.sort_values("年份", ascending=True)
year_list = df_map["年份"].unique().tolist()

heat_pieces = [
    {"min": 10000, "max": 20000, "label": "1万-2万元", "color": "#B2EBF2"},
    {"min": 20000, "max": 30000, "label": "2万-3万元", "color": "#80DEEA"},
    {"min": 30000, "max": 40000, "label": "3万-4万元", "color": "#4DD0E1"},
    {"min": 40000, "max": 50000, "label": "4万-5万元", "color": "#26C6DA"},
    {"min": 50000, "max": 70000, "label": "5万-7万元", "color": "#00ACC1"},
    {"min": 70000, "max": 100000, "label": "7万元以上", "color": "#0097A7"},
]

tl_income_map = Timeline(init_opts=opts.InitOpts(width=GLOBAL_WIDTH, height=GLOBAL_HEIGHT, theme=GLOBAL_THEME))
for year in year_list:
    year_data = df_map[df_map["年份"] == year]
    map_data = list(zip(year_data["省份名称"], year_data["人均可支配收入（元）"]))

    map_chart = (
        Map(init_opts=opts.InitOpts(width=GLOBAL_WIDTH, height=GLOBAL_HEIGHT, theme=GLOBAL_THEME))
        .add(
            series_name=f"{year}年人均可支配收入",
            data_pair=map_data,
            maptype="china",
            label_opts=opts.LabelOpts(is_show=True, font_size=LABEL_FONT_SIZE, color="#333"),
            is_map_symbol_show=False,
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(
                title=f"全国各省份{year}年人均可支配收入地理分布",
                subtitle="数据来源：近10年各省份居民收支与人口数据",
                pos_left="center",
                title_textstyle_opts=opts.TextStyleOpts(font_size=TITLE_FONT_SIZE, color=TITLE_COLOR),
                subtitle_textstyle_opts=opts.TextStyleOpts(font_size=LABEL_FONT_SIZE, color="#666")
            ),
            visualmap_opts=opts.VisualMapOpts(
                is_piecewise=True,
                pieces=heat_pieces,
                pos_left="left",
                pos_bottom="center",
                textstyle_opts=opts.TextStyleOpts(font_size=LABEL_FONT_SIZE),
                item_width=20,
                item_height=15
            ),
            tooltip_opts=opts.TooltipOpts(trigger="item", formatter="{b}<br/>人均可支配收入：{c} 元"),
            legend_opts=opts.LegendOpts(is_show=False)
        )
    )
    tl_income_map.add(map_chart, f"{year}年")

tl_income_map.add_schema(
    is_auto_play=True,
    play_interval=1500,
    is_loop_play=True,
    pos_left="center",
    pos_bottom="10px",
    width="80%",
    label_opts=opts.LabelOpts(font_size=LABEL_FONT_SIZE)
)

# ===================== 3. 面积图：收支趋势 =====================
year_mean = df.groupby('年份')[['人均可支配收入（元）', '人均消费支出（元）']].mean().reset_index()

line = (
    Line(init_opts=opts.InitOpts(width=GLOBAL_WIDTH, height=GLOBAL_HEIGHT, theme=GLOBAL_THEME))
    .add_xaxis([str(y) for y in year_mean['年份']])
    .add_yaxis(
        "人均可支配收入",
        year_mean['人均可支配收入（元）'].tolist(),
        areastyle_opts=opts.AreaStyleOpts(opacity=0.5),
        is_smooth=True
    )
    .add_yaxis(
        "人均消费支出",
        year_mean['人均消费支出（元）'].tolist(),
        areastyle_opts=opts.AreaStyleOpts(opacity=0.4),
        is_smooth=True
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="2016-2025全国居民收支趋势面积图",
            pos_left="center",
            title_textstyle_opts=opts.TextStyleOpts(font_size=TITLE_FONT_SIZE, color=TITLE_COLOR)
        ),
        tooltip_opts=TOOLTIP_STYLE,
        xaxis_opts=opts.AxisOpts(
            name="年份",
            axislabel_opts=opts.LabelOpts(font_size=AXIS_FONT_SIZE)
        ),
        yaxis_opts=opts.AxisOpts(
            name="金额（元）",
            axislabel_opts=opts.LabelOpts(font_size=AXIS_FONT_SIZE)
        ),
    )
)

# ===================== 4. 饼图：人口规模等级分布 =====================
df_2025 = df[df['年份'] == 2025].copy()

def classify_pop(pop):
    if pop >= 8000:
        return "超大型省份"
    elif pop >= 5000:
        return "大型省份"
    elif pop >= 2000:
        return "中型省份"
    else:
        return "小型省份"

df_2025['人口等级'] = df_2025['常住人口（万人）'].apply(classify_pop)
pop_class = df_2025['人口等级'].value_counts()
pie_data = list(zip(pop_class.index.tolist(), pop_class.values.tolist()))

pie = (
    Pie(init_opts=opts.InitOpts(width=GLOBAL_WIDTH, height=GLOBAL_HEIGHT, theme=GLOBAL_THEME))
    .add(
        series_name="省份数量",
        data_pair=pie_data,
        radius=["30%", "70%"],
        label_opts=opts.LabelOpts(formatter="{b}: {c}个 ({d}%)", font_size=LABEL_FONT_SIZE)
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="2025年各省份常住人口规模等级分布",
            pos_left="center",
            title_textstyle_opts=opts.TextStyleOpts(font_size=TITLE_FONT_SIZE, color=TITLE_COLOR)
        ),
        legend_opts=opts.LegendOpts(pos_left="left", orient="vertical", textstyle_opts=opts.TextStyleOpts(font_size=LABEL_FONT_SIZE)),
        tooltip_opts=TOOLTIP_STYLE
    )
)

# ===================== 5. 散点图：收入消费关系 =====================
df_scatter = df.copy()
df_scatter["消费率"] = df_scatter["人均消费支出（元）"] / df_scatter["人均可支配收入（元）"]

data = []
for _, row in df_scatter.iterrows():
    data.append([
        float(row["人均可支配收入（元）"]),
        float(row["人均消费支出（元）"]),
        float(row["消费率"])
    ])

scatter = (
    Scatter(init_opts=opts.InitOpts(width=GLOBAL_WIDTH, height=GLOBAL_HEIGHT, theme=GLOBAL_THEME))
    .add_xaxis([])
    .add_yaxis(
        series_name="收入-消费",
        y_axis=data,
        symbol_size=14,
        label_opts=opts.LabelOpts(is_show=False),
        itemstyle_opts=opts.ItemStyleOpts(opacity=0.85)
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="居民收入与消费支出关系（双维度映射）",
            title_textstyle_opts=opts.TextStyleOpts(font_size=TITLE_FONT_SIZE, color=TITLE_COLOR)
        ),
        xaxis_opts=opts.AxisOpts(
            name="人均可支配收入（元）", type_="value",
            axislabel_opts=opts.LabelOpts(font_size=AXIS_FONT_SIZE)
        ),
        yaxis_opts=opts.AxisOpts(
            name="人均消费支出（元）", type_="value",
            axislabel_opts=opts.LabelOpts(font_size=AXIS_FONT_SIZE)
        ),
        visualmap_opts=opts.VisualMapOpts(
            type_="color",
            min_=df_scatter["消费率"].min(),
            max_=df_scatter["消费率"].max(),
            dimension=2,
            range_color=["#457B9D", "#A8DADC", "#F1FAEE", "#E63946"],
            range_text=[f"{df_scatter['消费率'].max():.1%}", f"{df_scatter['消费率'].min():.1%}"],
            textstyle_opts=opts.TextStyleOpts(font_size=LABEL_FONT_SIZE)
        ),
        tooltip_opts=opts.TooltipOpts(
            trigger="item",
            formatter=JsCode("""
                function(params) {
                    let income = params.data[0];
                    let consume = params.data[1];
                    let rate = params.data[2];
                    return  '收入：' + income.toFixed(0) + ' 元<br/>' +
                            '消费：' + consume.toFixed(0) + ' 元<br/>' +
                            '消费率：' + (rate*100).toFixed(2) + '%';
                }
            """)
        )
    )
)

# ===================== 6. 柱状图：省份年均收入对比 =====================
avg_income = df.groupby("省份名称")["人均可支配收入（元）"].mean().reset_index()
avg_income = avg_income.sort_values("人均可支配收入（元）", ascending=False)

provinces = avg_income["省份名称"].tolist()
income_values = avg_income["人均可支配收入（元）"].round(2).tolist()

bar_avg = Bar(init_opts=opts.InitOpts(width=GLOBAL_WIDTH, height=GLOBAL_HEIGHT, theme=GLOBAL_THEME))
bar_avg.add_xaxis(provinces)
bar_avg.add_yaxis(
    "年均可支配收入（元）",
    income_values
)

bar_avg.set_global_opts(
    title_opts=opts.TitleOpts(
        title="全国各省份年均收入对比",
        title_textstyle_opts=opts.TextStyleOpts(font_size=TITLE_FONT_SIZE, color=TITLE_COLOR)
    ),
    xaxis_opts=opts.AxisOpts(
        axislabel_opts=opts.LabelOpts(rotate=45, font_size=AXIS_FONT_SIZE)
    ),
    yaxis_opts=opts.AxisOpts(
        name="收入（元）",
        name_textstyle_opts=opts.TextStyleOpts(font_size=AXIS_FONT_SIZE)
    ),
    datazoom_opts=[opts.DataZoomOpts(type_="inside")],
    tooltip_opts=TOOLTIP_STYLE
)

bar_avg.set_series_opts(label_opts=opts.LabelOpts(is_show=False))

# ===================== 聚合所有图表，生成大屏 =====================
page = Page(layout=Page.SimplePageLayout)
page.add(timeline_bar, tl_income_map, line, pie, scatter, bar_avg)
page.render("../实训/全国居民收支数据可视化大屏.html")

'C:\\Users\\Administrator\\PycharmProjects\\shujukeshihua\\实训\\全国居民收支数据可视化大屏.html'